# Домашнее задание: Binary Search on Answer на реальных данных

### Что нужно сделать

1. Найти **открытый датасет минимум на 100 000 строк** (будет супер, если какую-то сводку с помощью pandas (и мб numpy) ну и вашего текста вы напишете, чтобы я немного окунулась в сам датасет)
2. Разобраться, что означают его числовые признаки и какие особенности есть в данных
3. На **одном и том же датасете** придумать **минимум 3 разные постановки**, которые можно решать с помощью Binary Search on Answer
4. Для каждой постановки описать:
   - что требуется найти
   - что является `X`
   - что проверяет `check(X)`
   - направление монотонности и почему
   - границы `L` и `R`
5. Каждую из постановок полностью реализовать (будет поощряться, если вы потестите разные по сложности постановки, чтобы больше погрузиться в механику)
6. Для каждой реализованной задачи проверить, что найдено именно граничное значение
7. Для одной задачи провести эксперимент по времени работы на данных разного размера
8. Сделать короткие выводы

> **Важно:** три задачи должны отличаться не только числами. Нужны разные интерпретации одних и тех же данных

> Если в выбранных признаках есть отрицательные значения, нули, пропуски, выбросы или другие особенности — **не нужно автоматически их удалять**. Решите сами, как с ними работать, и обоснуйте это (мне без разницы, но как дата инженеры вы должны обдумать, как поступить лучше всего)

> Если `X` вещественный, укажите точность поиска `ε` и объясните, как вы останавливаете бинарный поиск. Если можно сформулировать задачу с целочисленным `X`, это обычно проще



## 0. Информация о работе

**ФИО:**  
Шипилов Алексей Константинович

**Группа:**  
11-511

**Ссылка на датасет:**  
https://excelbianalytics.com/downloads-18-sample-csv-files-data-sets-for-testing-sales/$0

**Источник датасета:**  
ExcelBI

**Кратко: что содержится в датасете?**  
Этот датасет представляет собой журнал международных продаж потребительских товаров, разбитый по регионам, странам и типам продукции. Он содержит детальную информацию о каждом заказе, включая каналы сбыта (онлайн/офлайн), даты оформления и доставки, а также количество проданных единиц. Наличие финансовых метрик (выручки, себестоимости и чистой прибыли) делает этот набор данных отличным инструментом для анализа коммерческой эффективности и логистики.

# 1. Загрузка датасета


In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv("sample_data/100000 Sales Records.csv")

### Проверка размера датасета

In [6]:
print("Размер датасета:", df.shape)

if len(df) < 100_000:
    print("ВНИМАНИЕ: в датасете меньше 100 000 строк. Анна Дмитриевна расстроится((((")
else:
    print("Требование по размеру выполнено. Анна Дмитриевна довольна")

Размер датасета: (100000, 14)
Требование по размеру выполнено. Анна Дмитриевна довольна


# 2. Быстрый анализ данных

Покажите, что вы понимаете, с какими данными работаете.

Нужно:

- посмотреть первые строки;
- изучить типы столбцов;
- выбрать один или несколько числовых признаков;
- проверить пропуски;
- заметить возможные отрицательные значения, нули, выбросы и другие особенности.

In [7]:
df.head()

,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Middle East and North Africa,Azerbaijan,Snacks,Online,C,10/8/2014,535113847,10/23/2014,934,152.58,97.44,142509.72,91008.96,51500.76
1,Central America and the Caribbean,Panama,Cosmetics,Offline,L,2/22/2015,874708545,2/27/2015,4551,437.20,263.33,1989697.20,1198414.83,791282.37
2,Sub-Saharan Africa,Sao Tome and Principe,Fruits,Offline,M,12/9/2015,854349935,1/18/2016,9986,9.33,6.92,93169.38,69103.12,24066.26
3,Sub-Saharan Africa,Sao Tome and Principe,Personal Care,Online,M,9/17/2014,892836844,10/12/2014,9118,81.73,56.67,745214.14,516717.06,228497.08
4,Central America and the Caribbean,Belize,Household,Offline,H,2/4/2010,129280602,3/5/2010,5858,668.27,502.54,3914725.66,2943879.32,970846.34


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Region          100000 non-null  object 
 1   Country         100000 non-null  object 
 2   Item Type       100000 non-null  object 
 3   Sales Channel   100000 non-null  object 
 4   Order Priority  100000 non-null  object 
 5   Order Date      100000 non-null  object 
 6   Order ID        100000 non-null  int64  
 7   Ship Date       100000 non-null  object 
 8   Units Sold      100000 non-null  int64  
 9   Unit Price      100000 non-null  float64
 10  Unit Cost       100000 non-null  float64
 11  Total Revenue   100000 non-null  float64
 12  Total Cost      100000 non-null  float64
 13  Total Profit    100000 non-null  float64
dtypes: float64(5), int64(2), object(7)
memory usage: 10.7+ MB


In [13]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print(*numeric_cols, sep=", ")

Order ID, Units Sold, Unit Price, Unit Cost, Total Revenue, Total Cost, Total Profit


In [14]:
df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Order ID,100000.0,5.503956e+08,2.593219e+08,1.000089e+08,3.260464e+08,5.477185e+08,7.750785e+08,9.999965e+08
Units Sold,100000.0,5.001446e+03,2.884575e+03,1.000000e+00,2.505000e+03,5.007000e+03,7.495250e+03,1.000000e+04
Unit Price,100000.0,2.667040e+02,2.169401e+02,9.330000e+00,1.092800e+02,2.057000e+02,4.372000e+02,6.682700e+02
Unit Cost,100000.0,1.880197e+02,1.757060e+02,6.920000e+00,5.667000e+01,1.171100e+02,3.646900e+02,5.249600e+02
Total Revenue,100000.0,1.336067e+06,1.471768e+06,1.866000e+01,2.797533e+05,7.898916e+05,1.836490e+06,6.682700e+06
Total Cost,100000.0,9.419755e+05,1.151828e+06,1.384000e+01,1.629283e+05,4.679374e+05,1.209475e+06,5.249075e+06
Total Profit,100000.0,3.940912e+05,3.795986e+05,4.820000e+00,9.590000e+04,2.836575e+05,5.683841e+05,1.738700e+06


In [15]:
df[numeric_cols].isna().sum().sort_values(ascending=False).head(20)

,0
Order ID,0
Units Sold,0
Unit Price,0
Unit Cost,0
Total Revenue,0
Total Cost,0
Total Profit,0


### Какие признаки вы выбрали?

**Признак(и):**  
Total Revenue, Total Cost, Total Profit, Order ID

**Что они означают?**  
Total Revenue - общий доход от продажи. Total Cost - все расходы на продажи. Total Profit - итоговая выручка от продажи. Order ID - ID заказа

**Какие особенности данных обнаружены?**  
Order ID - уникальный столбец для всех продаж. Total Revenue, Total Cost, Total Profit взаимосвязаны: Total Profit = Total Revenue - Total Cost

**Как вы решили с ними работать и почему?**  
Никаких выпадающих, пустых и неправильных значений не было, поэтому оставляем столбцы, как есть.

# 3. Постановка №1

### 3.1. Что требуется найти?

Компания хочет получить суммарную прибыль не менее (P) долларов, используя минимальное количество заказов.

Из датасета дан массив Total Profit, содержащий прибыль по каждому заказу.

Необходимо определить минимальное количество наиболее прибыльных заказов, суммарная прибыль которых будет не меньше (P).

### 3.2. Что является `X`?

`X = количество наиболее прибыльных заказов, необходимых для достижения целевой прибыли (P).`

### 3.3. Что проверяет `check(X)`?

`check(X) = Проверяет, достигает ли суммарная прибыль от X наиболее прибыльных заказов заданной целевой прибыли (P).`

### 3.4. Направление монотонности

- `FFFF | TTTT`

**Почему?**  
При увеличении X суммарная прибыль не уменьшается, поскольку в расчёт добавляются следующие наиболее прибыльные заказы (при условии, что прибыли неотрицательные). Поэтому после первого True все последующие проверки также будут True.

### 3.5. Границы поиска

`L = 0`

`R = 100000`

В датасете 100 000 заказов. Минимально можно выбрать 0 заказов, максимально — все 100 000. Если целевая прибыль \(P\) положительна и не превышает общую прибыль всех заказов, решение гарантированно существует в этих границах.

### 3.6. Почему эта постановка имеет смысл для выбранного датасета?

Датасет содержит информацию о продажах, включая показатель Total Profit. Это позволяет определить, какое минимальное количество наиболее прибыльных заказов необходимо для достижения заданной финансовой цели. Такая задача полезна для анализа эффективности продаж и выявления наиболее прибыльных заказов.

# 4. Постановка №2

### 4.1. Что требуется найти?

Компания должна отгрузить все заказы за D дней. Каждый заказ содержит определённое количество товаров (Units Sold).

Заказы обрабатываются в исходном порядке. Один заказ нельзя разделять между днями.

Определите минимальную ежедневную пропускную способность склада C, при которой все заказы можно обработать не более чем за D дней.

### 4.2. Что является `X`?

`X = минимальная ежедневная пропускная способность склада (количество единиц товара, которое склад может обработать за один день), необходимая для отправки всех заказов за D дней.`

### 4.3. Что проверяет `check(X)`?

`check(X) = проверяет, можно ли обработать все заказы за не более чем D дней при ежедневной пропускной способности склада X.`

### 4.4. Направление монотонности

Выберите:

- `FFFF | TTTT`

**Почему?**  
При увеличении X склад может обрабатывать больше товаров за день, поэтому необходимое количество дней не увеличивается. Если при некотором X все заказы удаётся обработать за D дней, то при любом большем X условие также будет выполняться.

### 4.5. Границы поиска

`L = максимальное количество товаров в одном заказе (Units Sold).`

`R = общее количество товаров во всех заказах.`

**Почему ответ гарантированно находится между ними?**

### 4.6. Почему эта постановка имеет смысл для выбранного датасета?

Пропускная способность склада не может быть меньше самого большого заказа, поскольку заказы нельзя разделять между днями. При пропускной способности, равной сумме всех товаров, склад сможет обработать все заказы за один день. Следовательно, при D >= 1 минимальная допустимая пропускная способность гарантированно находится в диапазоне [L, R].

# 5. Постановка №3

### 5.1. Что требуется найти?

Для каждого заказа известны даты оформления Order Date и отправки Ship Date.

Компания хочет определить минимальное количество дней T, за которое успевает отправить не менее K% всех заказов.

Вычислите срок доставки для каждого заказа как разницу между Ship Date и Order Date.

### 5.2. Что является `X`?

`X = минимальное количество дней между оформлением заказа (Order Date) и его отправкой (Ship Date), за которое компания успевает отправить не менее K% всех заказов.`

### 5.3. Что проверяет `check(X)`?

`check(X) = проверяет, составляет ли доля заказов, отправленных за X дней или быстрее, не менее K% от общего количества заказов.`

### 5.4. Направление монотонности

Выберите:

- `FFFF | TTTT`

**Почему?**  
При увеличении X количество заказов, удовлетворяющих условию, не уменьшается, поскольку учитываются заказы с более длительным сроком отправки. Если при некотором X условие выполняется, то при любом большем X оно также будет выполняться.

### 5.5. Границы поиска

`L = 0`

`R = max(data) — максимальное количество дней между Order Date и Ship Date среди всех заказов.`

Срок отправки не может быть отрицательным, поэтому нижняя граница равна 0. При X = max(data) все заказы удовлетворяют условию, то есть их доля составляет 100%. Следовательно, при 0 < K <= 100 ответ гарантированно находится между L и R, если даты в датасете корректны.

### 5.6. Почему эта постановка имеет смысл для выбранного датасета?

Датасет содержит 100 000 записей о продажах, включая даты оформления (Order Date) и отправки (Ship Date) заказов. Эти данные позволяют рассчитать время обработки каждого заказа и определить минимальный срок, за который компания успевает отправить не менее 90% заказов. Такая задача полезна для оценки скорости обработки заказов, анализа логистики и улучшения качества обслуживания клиентов.

## Проверка разнообразия постановок

**Чем задача №1 отличается от задачи №2?**  
В задаче 1 мы ищем минимальное количество наиболее прибыльных заказов, необходимое для достижения целевой прибыли.

В задаче 2 мы ищем минимальную ежедневную пропускную способность склада, позволяющую обработать все заказы за заданное количество дней.

Таким образом, первая задача связана с анализом прибыли, а вторая - с оптимизацией складской логистики.

**Чем задача №2 отличается от задачи №3?**  
В задаче 2 определяется минимальная пропускная способность склада при ограничении на количество дней.

В задаче 3 определяется минимальный срок, за который отправляется не менее 90% заказов.

Вторая задача связана с расстановкой ресурсов склада, а третья — с проверкой того, как быстро отправятся товары.

**Есть ли среди ваших задач разные направления монотонности (`FFFF|TTTT` и `TTTT|FFFF`)?**  
Во всех трёх задачах используется монотонность FFFF | TTTT. Датасет построен на продажах. Поэтому для него более естественныи оказались задачи на поиск первого минимального временит, выручки и тп. Это соответствует реальным задачам на поиск в бизнесе)

# 6. Реализация первой выбранной задачи

Перед кодом зафиксируйте:

- `X = количество наиболее прибыльных заказов, необходимых для достижения целевой прибыли (P).`
- `check(X) = Проверяет, достигает ли суммарная прибыль от X наиболее прибыльных заказов заданной целевой прибыли (P).`
- монотонность = FFFF | TTTT
- `L = 0`
- `R = 100000`

In [17]:
import pandas as pd
import numpy as np

# Загружаем датасет
df = pd.read_csv("sample_data/100000 Sales Records.csv")

# Подготавливаем данные
# Сортируем прибыль по убыванию
data = np.sort(df["Total Profit"].dropna().to_numpy())[::-1]

# Целевая прибыль
P = 10_000_000_000

In [18]:
def check_1(x, data):
    """Проверяем, достигает ли сумма прибыли от x наиболее прибыльных заказов цели P."""

    if x < 0 or x > len(data):
        return False

    return np.sum(data[:x]) >= P

In [19]:
def solve_1(data):
    left = 0
    right = 100000

    # Проверка существования решения
    if not check_1(right, data):
        return None

    while left < right:
        mid = (left + right) // 2

        if check_1(mid, data):
            right = mid
        else:
            left = mid + 1

    return left

In [20]:
answer_1 = solve_1(data)

print("Ответ:", answer_1)

Ответ: 7420


### Проверка границы

Если вы ищете **первый `True`**, должно выполняться:

```text
check(answer) = True
check(answer - 1) = False
```

Если вы ищете **последний `True`**:

```text
check(answer) = True
check(answer + 1) = False
```

Если соседнее значение выходит за допустимый диапазон, объясните это

In [22]:
# Проверка границы
if answer_1 is not None:
  print("check(answer) =", check_1(answer_1, data))

  if answer_1 > 0:
    print("check(answer - 1) =", check_1(answer_1 - 1, data))
  else:
      print("answer - 1 вне диапазона")

  print("Суммарная прибыль:", round(np.sum(data[:answer_1]), 2))
else:
  print("Целевая прибыль недостижима")

check(answer) = True
check(answer - 1) = False
Суммарная прибыль: 10000646456.9


### Сложность решения №1

**Сложность `check(X)`:**  
**O(X)**, в худшем случае **O(N)**, так как функция np.sum(data[:x]) суммирует первые X элементов массива.

**Количество итераций бинарного поиска:**  
O(log N). При N = 100000 потребуется не более 17 итераций.

**Общая сложность:**  
O(N*log N)

**Дополнительная память:**  
O(N) для создания отсортированного массива. Сам бинарный поиск использует O(1) дополнительной памяти.

# 7. Реализация второй выбранной задачи

Теперь выберите **другую** постановку и реализуйте её полностью.

In [23]:
import pandas as pd
import numpy as np

# Загружаем датасет
df = pd.read_csv("sample_data/100000 Sales Records.csv")

# Получаем количество товаров в каждом заказе
data = df["Units Sold"].dropna().to_numpy()

# Получаем количество товаров в каждом заказе
data = df["Units Sold"].dropna().to_numpy()

# Максимальное количество дней
D = 100

In [24]:
def check_2(x, data):
    """Проверяем, можно ли обработать все заказы за D дней при пропускной способности x."""

    days = 1
    current_load = 0

    for units in data:
        if units > x:
            return False

        if current_load + units <= x:
            current_load += units
        else:
            days += 1
            current_load = units

            if days > D:
                return False

    return True

In [25]:
def solve_2(data):
    left = int(max(data))
    right = int(sum(data))

    # Проверка существования решения
    if not check_2(right, data):
        return None

    while left < right:
        mid = (left + right) // 2

        if check_2(mid, data):
            right = mid
        else:
            left = mid + 1

    return left

In [26]:
answer_2 = solve_2(data)

print("Ответ:", answer_2)

Ответ: 5004415


In [27]:
# Проверка границы
if answer_2 is not None:
  print("check(answer) =", check_2(answer_2, data))

  if answer_2 > max(data):
      print("check(answer - 1) =", check_2(answer_2 - 1, data))
  else:
      print("answer - 1 вне допустимого диапазона")
else:
  print("Решение не найдено")

check(answer) = True
check(answer - 1) = False


### Сложность решения №2

**Сложность `check(X)`:**  
O(N), поскольку функция последовательно обрабатывает все заказы, проверяя, сколько дней потребуется.

**Количество итераций бинарного поиска:**  
O(log(R-L+1)), где L = max(data), R = sum(data).

**Общая сложность:**  
O(N*log S), где S — суммарное количество товаров во всех заказах.

**Дополнительная память:**  
O(1) для бинарного поиска и проверки check(X), так как используются только несколько дополнительных переменных. Массив data занимает O(N) памяти.

# 8. Эксперимент на больших данных

Для **одной** из двух реализованных задач измерьте время работы на:

- 1 000 строк;
- 10 000 строк;
- 100 000 строк;
- полном датасете.

Для эксперимента выбрана задача 1 — поиск минимального количества наиболее прибыльных заказов.

In [32]:

import pandas as pd
import numpy as np
import time

df = pd.read_csv("sample_data/100000 Sales Records.csv")

data = np.sort(
    df["Total Profit"].dropna().to_numpy()
)[::-1]

def check_1(x, data, P):
    return np.sum(data[:x]) >= P

def solve_1(data):
    P = np.sum(data) * 0.5

    left = 0
    right = len(data)

    while left < right:
        mid = (left + right) // 2

        if check_1(mid, data, P):
            right = mid
        else:
            left = mid + 1

    return left


solve_fn = solve_1
full_data = data

sizes = [1_000, 10_000, 100_000, len(full_data)]
sizes = sorted(set(n for n in sizes if n <= len(full_data)))

results = []

for n in sizes:
    subset = full_data[:n]

    start = time.perf_counter()
    answer = solve_fn(subset)
    elapsed = time.perf_counter() - start

    results.append({
        "N": n,
        "answer": answer,
        "time_sec": elapsed
    })

timing_df = pd.DataFrame(results)
timing_df

,N,answer,time_sec
0,1000,490,0.000100
1,10000,4244,0.000085
2,100000,18974,0.000238


### Что получилось?

**Как меняется время работы при увеличении `N`?**  
При увеличении количества строк время выполнения алгоритма растёт. Но при 1000 строк, результат получился медленнее. Почему? Из-за накладных расходов на фоновые процессы компьютера, нагрузка на процессор, накладные расходы Python и NumPy, случайные колебания времени выполнения. Поэтому при 10000 результат получился быстрее.

**Соответствует ли это вашей теоретической оценке сложности?**  
Да. Теоретическая сложность поиска без префиксных сумм составляет O(N*log N). Полученные результаты согласуются с этой оценкой: время увеличивается с ростом размера данных.

**Что сильнее влияет на время: размер датасета или ширина диапазона `[L, R]`? Почему?**  
Сильнее влияет размер датасета N, поскольку проверка check(X) требует суммирования до X элементов. Ширина диапазона [L, R] влияет на количество итераций бинарного поиска только логарифмически. Поэтому увеличение количества обрабатываемых элементов обычно оказывает более существенное влияние на время выполнения.

# 9. Сравнение с перебором

Для той же задачи оцените, что было бы при простом переборе всех возможных `X`

Не обязательно запускать полный перебор на полном датасете

**Сколько примерно различных значений `X` пришлось бы проверить перебором?**  
В датасете 100 000 заказов, поэтому возможные значения X находятся в диапазоне от 0 до 100 000. При полном переборе пришлось бы проверить до 100 001 значения X.Если перебирать значения по возрастанию и останавливаться на первом True, количество проверок будет зависеть от найденного ответа.

**Сколько проверок делает Binary Search?**  
Бинарный поиск сокращает диапазон примерно в два раза на каждой итерации. При N = 100000 потребуется не более 17 итераций.

**Почему Binary Search on Answer здесь выигрывает?**  
Binary Search on Answer выигрывает, потому что вместо последовательной проверки всех возможных значений X он каждый раз уменьшает диапазон поиска вдвое.

При этом количество проверок сокращается с O(N) до O(log N).

In [33]:

import time

# Перебор всех возможных X
def brute_force(data):
    P = np.sum(data) * 0.5

    for x in range(len(data) + 1):
        if np.sum(data[:x]) >= P:
            return x

    return None


# Небольшая выборка
subset = full_data[:1000]

# Измерение времени перебора
start = time.perf_counter()
brute_answer = brute_force(subset)
brute_time = time.perf_counter() - start

# Измерение времени бинарного поиска
start = time.perf_counter()
binary_answer = solve_1(subset)
binary_time = time.perf_counter() - start

print("Перебор:")
print("Ответ:", brute_answer)
print("Время:", brute_time)

print("\nBinary Search:")
print("Ответ:", binary_answer)
print("Время:", binary_time)

print("\nОтветы совпадают:", brute_answer == binary_answer)


Перебор:
Ответ: 490
Время: 0.0021269619992381195

Binary Search:
Ответ: 490
Время: 0.0002075120000881725

Ответы совпадают: True


# 10. Выводы

### 1. Какая из трёх постановок оказалась самой естественной для этого датасета? Почему?

Самой естественной оказалась задача 1 — поиск минимального количества наиболее прибыльных заказов для достижения целевой прибыли. Потому что это прям реальная задача в бизнесе: найти самые прибыльные заказы среди всех заказов.

### 2. Какая постановка получилась наиболее искусственной? Почему?

Наверное, задача 2 - определение минимальной пропускной способности склада. Нам на указано, какова вместимость склада, реальных данных нет, и мы берем ее из головы чисто для условия задачи.

### 3. Что в Binary Search on Answer оказалось самым сложным?

- придумать `X`;
- придумать `check(X)`;
- доказать монотонность;
- подобрать `L/R`;
- написать бинарный поиск;
- другое.

Придумать check x и написать код

### 4. В каких случаях Binary Search on Answer здесь был действительно полезен?

Во всех случаях. Во всех задачах надо было найти какое-то минимальное значение из данных. Делать это все перебором в 10 раз медленнее, чем бинарным поиском.